# Comparison of scenarios

In [1]:
mt_rate = 0.1

output_name = 'taunton_worcester'

In [2]:
import pandas as pd
import matplotlib.pyplot as plt

In [3]:
base_path = "./output/results_full_current_hospitals.csv"
comparitor_path = "./output/results_full_taunton_and_worcester.csv"

base_data = pd.read_csv(base_path)
comparitor_data = pd.read_csv(comparitor_path)

hospitals = pd.read_csv("./data/stroke_hospitals_2025.csv", index_col='Postcode')

In [4]:
base_data.head()

,LSOA,nearest_ivt_unit,nearest_ivt_time,nearest_ivt_unit_no_bias,nearest_ivt_time_no_bias,nearest_mt_unit,nearest_mt_time,transfer_unit,transfer_required,transfer_time,...,lvo_mothership_ivt_mt_utility_shift,drip_ship_utility_shift,mothership_utility_shift,nlvo_drip_ship_ivt_improvement_over_no_treatment,lvo_drip_ship_ivt_improvement_over_no_treatment,no_treatment_mrs_0-2,drip_ship_mrs_0-2,drip_ship_mrs_0-2_improvement_over_no_treatment,mothership_mrs_0-2,mothership_mrs_0-2_improvement_over_no_treatment
0,ADUR001A,BN25BE,10,BN25BE,11,BN25BE,11,BN25BE,False,0,...,0.155584,0.114428,0.124652,0.120947,0.060172,0.4855,0.608202,0.122702,0.619167,0.133667
1,ADUR001B,BN25BE,11,BN25BE,12,BN25BE,12,BN25BE,False,0,...,0.154986,0.113964,0.124185,0.120497,0.059881,0.4855,0.607699,0.122199,0.618658,0.133158
2,ADUR001C,BN112DH,11,BN112DH,11,BN25BE,13,BN25BE,True,24,...,0.154388,0.109743,0.123719,0.120497,0.059881,0.4855,0.603213,0.117713,0.618149,0.132649
3,ADUR001D,BN25BE,11,BN25BE,12,BN25BE,12,BN25BE,False,0,...,0.154986,0.113964,0.124185,0.120497,0.059881,0.4855,0.607699,0.122199,0.618658,0.133158
4,ADUR001E,BN112DH,10,BN112DH,10,BN25BE,13,BN25BE,True,24,...,0.154388,0.110205,0.123719,0.120947,0.060172,0.4855,0.603713,0.118213,0.618149,0.132649


In [5]:
comparison = pd.DataFrame()
comparison['lsoa'] = base_data['area_lower']
comparison['admissions'] = base_data['Admissions_2025_Est']
comparison['base_ivt_unit'] = base_data['nearest_ivt_unit']
comparison['comparitor_ivt_unit'] = comparitor_data['nearest_ivt_unit']
comparison['base_mt_unit'] = base_data['nearest_mt_unit']
comparison['comparitor_mt_unit'] = comparitor_data['nearest_mt_unit']
comparison['base_transfer_in_MT'] = base_data['transfer_required']
comparison['comparitor_transfer_in_MT'] = comparitor_data['transfer_required']
comparison['base_ivt_time'] = base_data['nearest_ivt_unit_no_bias']
comparison['comparitor_ivt_time'] = comparitor_data['nearest_ivt_unit_no_bias']
comparison['base_mt_time'] = base_data['nearest_mt_time']
comparison['comparitor_mt_time'] = comparitor_data['nearest_mt_time']
comparison['base_mrs_0-2'] = base_data['drip_ship_mrs_0-2']
comparison['comparitor_mrs_0-2'] = comparitor_data['drip_ship_mrs_0-2']
comparison['changed'] = 1.0 * (comparison['comparitor_mrs_0-2'] > comparison['base_mrs_0-2'])


In [6]:
comparison.head()

,lsoa,admissions,base_ivt_unit,comparitor_ivt_unit,base_mt_unit,comparitor_mt_unit,base_transfer_in_MT,comparitor_transfer_in_MT,base_ivt_time,comparitor_ivt_time,base_mt_time,comparitor_mt_time,base_mrs_0-2,comparitor_mrs_0-2,changed
0,Adur 001A,1.49403,BN25BE,BN25BE,BN25BE,BN25BE,False,False,BN25BE,BN25BE,11,11,0.608202,0.608202,0.0
1,Adur 001B,1.86762,BN25BE,BN25BE,BN25BE,BN25BE,False,False,BN25BE,BN25BE,12,12,0.607699,0.607699,0.0
2,Adur 001C,2.24110,BN112DH,BN112DH,BN25BE,BN25BE,True,True,BN112DH,BN112DH,13,13,0.603213,0.603213,0.0
3,Adur 001D,2.24110,BN25BE,BN25BE,BN25BE,BN25BE,False,False,BN25BE,BN25BE,12,12,0.607699,0.607699,0.0
4,Adur 001E,4.10862,BN112DH,BN112DH,BN25BE,BN25BE,True,True,BN112DH,BN112DH,13,13,0.603713,0.603713,0.0


In [7]:
# Get unique base IVT units
base_ivt_units = comparison['base_ivt_unit'].unique().tolist()

# Set up the dataframe with base IVT units as the index
comparison_by_ivt_unit = pd.DataFrame(index=base_ivt_units)

# Add hospital details
comparison_by_ivt_unit['SSNAP_name'] = hospitals.loc[
    base_ivt_units, 'SSNAP name'
]
comparison_by_ivt_unit['Hospital_City'] = hospitals.loc[
    base_ivt_units, 'Hospital_City'
]

for unit in base_ivt_units:
    base_ivt = comparison['base_ivt_unit'] == unit
    comparator_ivt = comparison['comparitor_ivt_unit'] == unit
    base_mt = comparison['base_mt_unit'] == unit
    comparator_mt = comparison['comparitor_mt_unit'] == unit

    # First admissions
    base_admissions = comparison.loc[base_ivt, 'admissions'].sum()
    comparator_admissions = comparison.loc[
        comparator_ivt, 'admissions'
    ].sum()
    comparison_by_ivt_unit.loc[unit, 'base first admissions'] = int(
        base_admissions
    )
    comparison_by_ivt_unit.loc[
        unit, 'comparator first admissions'
    ] = int(comparator_admissions)

    # MT admissions
    base_mt_admissions = comparison.loc[base_mt, 'admissions'].sum()
    comparator_mt_admissions = comparison.loc[
        comparator_mt, 'admissions'
    ].sum()
    comparison_by_ivt_unit.loc[unit, 'base all MT admissions'] = int(
        base_mt_admissions * mt_rate
    )
    comparison_by_ivt_unit.loc[unit, 'comparator all MT admissions'] = int(
        comparator_mt_admissions * mt_rate
    )

    # Transfers in for MT
    comparison_by_ivt_unit.loc[unit, 'base_transfer_in_MT'] = int(
        comparison.loc[base_mt, 'base_transfer_in_MT'].sum() * mt_rate
    )
    comparison_by_ivt_unit.loc[unit, 'comparator_transfer_in_MT'] = int(
        comparison.loc[
            comparator_mt, 'comparitor_transfer_in_MT'
        ].sum() * mt_rate
    )

    # Outcomes weighted by admissions
    base_weighted_outcome = (
        comparison.loc[base_ivt, 'base_mrs_0-2']
        * comparison.loc[base_ivt, 'admissions']
    ).sum()
    comparator_weighted_outcome = (
        comparison.loc[comparator_ivt, 'comparitor_mrs_0-2']
        * comparison.loc[comparator_ivt, 'admissions']
    ).sum()

    base_denominator = comparison_by_ivt_unit.loc[
        unit, 'base first admissions'
    ]
    comparator_denominator = comparison_by_ivt_unit.loc[
        unit, 'comparator first admissions'
    ]

    comparison_by_ivt_unit.loc[unit, 'base outcome'] = (
        base_weighted_outcome / base_denominator
        if base_denominator
        else float('nan')
    )
    comparison_by_ivt_unit.loc[unit, 'comparator outcome'] = (
        comparator_weighted_outcome / comparator_denominator
        if comparator_denominator
        else float('nan')
    )

    # Changes between scenarios
    comparison_by_ivt_unit.loc[unit, 'change in first admissions'] = (
        comparison_by_ivt_unit.loc[unit, 'comparator first admissions']
        - comparison_by_ivt_unit.loc[unit, 'base first admissions']
    )
    comparison_by_ivt_unit.loc[unit, 'change in all MT admissions'] = (
        comparison_by_ivt_unit.loc[unit, 'comparator all MT admissions']
        - comparison_by_ivt_unit.loc[unit, 'base all MT admissions']
    )
    comparison_by_ivt_unit.loc[unit, 'change in outcome'] = (
        comparison_by_ivt_unit.loc[unit, 'comparator outcome']
        - comparison_by_ivt_unit.loc[unit, 'base outcome']
    )

comparison_by_ivt_unit.to_csv(
    f'./output/comparison_by_ivt_unit_{output_name}.csv',
    index=True,
)

In [8]:
comparison_by_ivt_unit.head(10)

,SSNAP_name,Hospital_City,base first admissions,comparator first admissions,base all MT admissions,comparator all MT admissions,base_transfer_in_MT,comparator_transfer_in_MT,base outcome,comparator outcome,change in first admissions,change in all MT admissions,change in outcome
BN25BE,Royal Sussex County Hospital,Brighton,668.0,668.0,255.0,255.0,65.0,65.0,0.607950,0.607950,0.0,0.0,0.0
BN112DH,Worthing Hospital,Worthing,557.0,557.0,0.0,0.0,0.0,0.0,0.604716,0.604716,0.0,0.0,0.0
CA27HY,Cumberland Infirmary,Carlisle,679.0,679.0,0.0,0.0,0.0,0.0,0.588911,0.588911,0.0,0.0,0.0
NG174JL,Kings Mill Hospital,Mansfield,738.0,738.0,0.0,0.0,0.0,0.0,0.604727,0.604727,0.0,0.0,0.0
DE223NE,Royal Derby Hospital,Derby,1035.0,1035.0,0.0,0.0,0.0,0.0,0.603740,0.603740,0.0,0.0,0.0
NG72UH,Queen's Medical Centre - Nottingham,Nottingham,1418.0,1418.0,437.0,437.0,101.0,101.0,0.607493,0.607493,0.0,0.0,0.0
PO196SE,St Richards Hospital,Chichester,518.0,518.0,0.0,0.0,0.0,0.0,0.601058,0.601058,0.0,0.0,0.0
CT13NG,East Kent HASU,Canterbury,1221.0,1221.0,201.0,201.0,35.0,35.0,0.603147,0.603147,0.0,0.0,0.0
ME169QQ,Maidstone District General Hospital,Maidstone,988.0,988.0,0.0,0.0,0.0,0.0,0.599442,0.599442,0.0,0.0,0.0
MK65LD,Milton Keynes General Hospital,Milton Keynes,706.0,706.0,0.0,0.0,0.0,0.0,0.598460,0.598460,0.0,0.0,0.0


In [9]:
# Get units with a change in MT admissions
mask = comparison_by_ivt_unit['change in all MT admissions'] != 0
units_with_change = comparison_by_ivt_unit.loc[mask]
units_with_change.head(20)

,SSNAP_name,Hospital_City,base first admissions,comparator first admissions,base all MT admissions,comparator all MT admissions,base_transfer_in_MT,comparator_transfer_in_MT,base outcome,comparator outcome,change in first admissions,change in all MT admissions,change in outcome
OX39DU,John Radcliffe Hospital,Oxford,838.0,838.0,223.0,218.0,59.0,58.0,0.604474,0.604474,0.0,-5.0,0.000000
BS105NB,Bristol and Weston SU,North Bristol,1397.0,1388.0,436.0,272.0,103.0,53.0,0.605151,0.605543,-9.0,-164.0,0.000392
B152TH,Queen Elizabeth Hospital Edgbaston,Birmingham,862.0,861.0,609.0,485.0,181.0,142.0,0.607684,0.607345,-1.0,-124.0,-0.000339
CV22DX,University Hospital Coventry,Coventry,1522.0,1514.0,389.0,385.0,92.0,91.0,0.606604,0.606407,-8.0,-4.0,-0.000196
WR51DD,Worcestershire Royal Hospital,Worcester,690.0,705.0,0.0,202.0,0.0,43.0,0.601132,0.606675,15.0,202.0,0.005543
ST46QG,Royal Stoke University Hospital,Stoke,1093.0,1093.0,232.0,229.0,37.0,36.0,0.607875,0.607875,0.0,-3.0,0.000000
PL68DH,Derriford Hospital,Plymouth,827.0,827.0,348.0,243.0,75.0,45.0,0.606048,0.606048,0.0,-105.0,0.000000
TA15DA,Musgrove Park Hospital,Taunton,927.0,942.0,0.0,246.0,0.0,41.0,0.593584,0.603645,15.0,246.0,0.010061
SO166YD,Southampton General Hospital,Southampton,828.0,828.0,555.0,525.0,141.0,134.0,0.608731,0.608731,0.0,-30.0,0.000000
